<img src="../../images/SnowparkIconLabel.png" alt="Snowpark Icon" width=150px align=right /> 

# Python Vectorized UDFs 

Vectorized Python UDFs let you define Python functions that receive batches of input rows as Pandas `DataFrame`s and return batches of results as Pandas arrays or `Series`. You call vectorized Python UDFs the same way you call other Python UDFs. Use vectorized UDFs as an alternative to the default approach of row-by-row processing.

**Important:** This lesson assumes prior knowledge of Snowflake for Python UDFs & UDTFs. See lesson [*Creating and Registering User-Defined Functions (UDFs)*](../Lectures/06-Creating-and-Registering-UDFs.ipynb)

#### Advantages
- The potential for better performance if your Python code operates efficiently on batches of rows.
- Less transformation logic required if you are using libraries that operate on Pandas `DataFrames` or Pandas arrays.
- You do not need to change how you write code using Python UDFs. The batching is handled by the Snowflake UDF framework and not specified in your own code.

#### Caveats
- There is no guarantee of which instances of your handler code will see which batch(es) of input. This is true with non-vectorized UDFs as well. 

> **&#128221; Note:** See documentation for further details:
> - [Vectorized Python UDFs](https://docs.snowflake.com/en/developer-guide/udf/python/udf-python-batch)
> - [Vectorized Python UDTFs](https://docs.snowflake.com/en/developer-guide/udf/python/udf-python-tabular-vectorized)

## Topics in this Lesson

1. [Proper Testing](#proper_testing)  
    1. [Disabling Query Result Cache Use ](#proper_testing)  
    1. [Preventing Skew from Warehouse Data Cache](#wh_data_cache)  
1. [Create a Non-Vectorized UDF and Use It in A Transformation](#create_non_vectorized_udf)  
    1. [The Steps](#the_steps_non_vectorized_udf)  
1. [Test A Non-Vectorized UDF and Measure Performance](#test_non_vectorized_udf)  
1. [Create a Vectorized UDF and Use It in A Transformation](#create_vectorized_udf)  
    1. [Option A: Use the `vectorized` Decorator](#option_a_decorator)  
        1. [The Steps (Option A)](#steps_option_a)  
        1. [Test the Vectorized UDF and Measure Performance (Option A)](#test_option_a)    
        1. [Compare Vectorized Performance (Option A)](#comp_perf_option_a)  
    1. [Option B: Use Function Attributes](#option_b_func_attributes)  
        1. [The Steps (Option B)](#steps_option_b)  
        1. [Test the Vectorized UDF and Measure Performance (Option B)](#test_option_b)  
        1. [Compare Vectorized Performance (Option B)](#comp_perf_option_b)  
    1. [Using `pandas.Series` vs `pandas.DataFrame`](#series_instead_of_dataframe)  
        1. [The Steps (Series)](#steps_series)  
        1. [Test the Vectorized UDF and Measure Performance (Series)](#test_series)   
        1. [Compare Vectorized Performance (`panda.Series`)](#comp_perf_series)  
1. [Clean Up and Close Session](#AV_cleanup)

#### Connect and create a `Session` 
1. Import required libraries.
1. Create a `Session` to connect to Snowflake.  

    > &#10071; Success requires that you have already completed the key pair authentication exercise.
1. Set context items for this module.

Run the following cells to connect to your Snowflake account. *You needn't edit anything in the following cells. Just run them.*

In [ ]:
# Run utils notebook
%run ../../utils/ds_utils_python.ipynb

# Connect to Snowflake and create a Session object named session
session = create_session()

---
#### Setup Code

We should set ourselves up for success. The following ensures our context is set properly for database, schema, role, and warehouse. *You needn't edit anything in the following cell. Just run it.*  Give it a few seconds to complete. When finished, you should see the message:
```
========== READY ===========
```

In [ ]:
# Hard code the lesson name
lesson_name = "VECTORIZED_UDFS"

# Create the context items for this lesson
lesson = confirm_or_create_lesson_context(session, lesson_name)

---

Our warehouse will need to be a medium for demo purposes. 

In [ ]:
wh_name = lesson.warehouse()
session.sql(f"ALTER WAREHOUSE {wh_name} SET WAREHOUSE_SIZE = MEDIUM").show()

---

<a id="proper_testing"></a>
## 1. Proper Testing

In this lesson, performance is compared between non-vectorized UDFs and vectorized UDFs. To glean accurate performance metrics we will disable two of Snowflake's built-in features for accelerating performance. 

<a id="qr_cache"></a>
### 1A. Disabling Query Result Cache Use 

Snowflake caches the result sets of statements in the Cloud Services layer of your Snowflake account. This is called Query Result Cache or "QR Cache". QR Cache speeds up performance if the same query is executed within a 24-hour period and the underlying data on which the query operates has not changed. 

From the [docs](https://docs.snowflake.com/en/user-guide/querying-persisted-results):
> *When a query is executed, the result is persisted (i.e. cached) for a period of time. At the end of the time period, the result is purged from the system.*
> 
> *Snowflake uses persisted query results to avoid re-generating results when nothing has changed (i.e. “retrieval optimization”). In addition, you can use persisted query results to post-process the results (e.g. layering a new query on top of the results already calculated).*
> 
> *For persisted query results of all sizes, the cache expires after 24 hours.*

To compare compute times between different approaches to data processing, we need to disable the use of the QR Cache. We can achieve this using the parameter `USE_CACHED_RESULT` and setting it to `FALSE`. This parameter can be set at several levels: Account, User, and Session. For this lesson we will alter our session **not** to use QR Cache. 

The Snowpark for Python API does not have a built-in function for altering a Snowflake session. To alter this parameter and `ALTER SESSION` statement is needed. This `session.sql(....)` is needed. 

```sql
...
ALTER SESSION SET USE_CACHED_RESULT = FALSE;
...
```

> **&#128221; Note:** See documentation for further details:
> - [Using Persisted Query Results](https://docs.snowflake.com/en/user-guide/querying-persisted-results)
> - [Snowflake Parameter USE_CACHED_RESULT](https://docs.snowflake.com/en/sql-reference/parameters#use-cached-result)

In [ ]:
# Alter our Snowflake session
session.sql("ALTER SESSION SET USE_CACHED_RESULT = FALSE").show()

# Confirm the parameter value has been changed
session.sql("SHOW PARAMETERS LIKE 'USE_CACHED_RESULT' IN SESSION").show()

<a id="wh_data_cache"></a>
### 1B. Preventing Skew from Warehouse Data Cache

When a Snowflake Virtual Warehouse reads data from storage, that data is cached on the SSDs in that VWH. Subsequent queries executed by that warehouse can take advantage of this cache to improve performance. 

To compare compute times between different approaches to data processing, we need to disable the use of the warehouse data cache. We can achieve this by suspending and then resuming the warehouse before each test query. The goal here is to present a level playing field for query performance comparisons so we can make good decisions regarding performance enhancements. 

From the [docs](https://docs.snowflake.com/en/user-guide/warehouses-considerations#how-does-warehouse-caching-impact-queries):
> This cache is dropped when the warehouse is suspended, which may result in slower initial performance for some queries after the warehouse is resumed. As the resumed warehouse runs and processes more queries, the cache is rebuilt, and queries that are able to take advantage of the cache will experience improved performance.

Below is function named `bounce_warehouse()` that will suspend and then resume the current warehouse. 

> **&#128221; Note:** See documentation for further details:
> - Snowflake docs: [How Does Warehouse Caching Impact Queries?](https://docs.snowflake.com/en/user-guide/warehouses-considerations#how-does-warehouse-caching-impact-queries)

In [ ]:
def bounce_warehouse(_session=session, confirm=False) -> None:
    
    """ Suspends the session's current warehouse and then resumes it. 
    This clears all data cache from the SSDs so testing can be fair and accurate
    
    @param: confirm Print confirmation messages - default is False
    
    """
    
    # Retrieve the current warehouse name from the session object
    curr_wh = _session.get_current_warehouse()
    
    # Attempt to suspend the warehouse
    try:
        # Suspending a warehouse that is not resumed can raise and exception
        _session.sql(f"ALTER WAREHOUSE {curr_wh} SUSPEND").collect()        
    except Exception as ex:
        pass # Ignore the exception
    
    # Attempt to resume the warehouse
    # The clause IF SUSPENDED prevents an exception being raised if the warehouse is already resumed
    _session.sql(f"ALTER WAREHOUSE {curr_wh} RESUME IF SUSPENDED").collect()

    # Print current warehouse state
    curr_wh = curr_wh.strip('"') # Get rid of the double quote marks
    
    # Print a confirmation if caller desired it
    if(confirm): print(f"Warehouse {curr_wh} bounced!")
    
    from snowflake.snowpark.types import BooleanType
    from snowflake.snowpark.functions import col
    
    # Print brief info on the warehouse if caller desired
    if(confirm): (_session
         .sql(f"SHOW WAREHOUSES LIKE '{curr_wh}'")
         .select(
              col('"name"').as_("Warehouse Name")
             ,col('"state"').as_("Current State")
             ,col('"size"').as_("Current Size")
             ,col('"running"').as_("Number of running queries")
          ) 
         .show()
    )
    return None

print("Function bounce_warehouse(Session,bool) created")

# Invoke it
bounce_warehouse(confirm=True)

---

We need a `DataFrame` for demonstration and testing purposes. It should represent a modest to hefty amount of data for demonstrating distinct performance gains using vectorized UDFs. 

In [ ]:
data_schema = "TPCH_SF1000"
from snowflake.snowpark.functions import col

parts_df = session.table(f"TRAINING_DB.{data_schema}.PART")
    
parts_supp_df = session.table(f"TRAINING_DB.{data_schema}.PARTSUPP")
    
joined_df = (
     parts_df.join(parts_supp_df,
            parts_df["P_PARTKEY"]==parts_supp_df["PS_PARTKEY"])
)
    
cost_and_price_df = (joined_df.select(
         col("P_PARTKEY").as_("part_id")
        ,col("PS_SUPPKEY").as_("supplier_id")
        ,col("P_RETAILPRICE").as_("retail_price") 
        ,col("PS_SUPPLYCOST").as_("cost")        
    )
)
print(f"cost_and_price_df created for schema {data_schema}")

---
<a id="create_non_vectorized_udf"></a> 

## 2. Create a Non-Vectorized UDF and Use in a Transformation

<a id="the_steps_non_vectorized_udf"></a>
#### The Steps
1. Create a simple function that calculates the margin on an order by subtracting the item cost from the item price. 
1. Register it as a **non-vectorized** Snowflake for Python UDF using `functions.udf(...)`.
1. Create a new `DataFrame` by using our non-vectorized UDF in a transformation. 

In [ ]:
# A Python function to power a non-vectorized UDF
def margin_func(price,cost):
    return price-cost
print("Python function margin_func created")

# Create and register the UDF object
from snowflake.snowpark.functions import udf
from snowflake.snowpark.types import DoubleType
margin_udf = (
    udf(
         func=margin_func
        ,return_type=DoubleType()
        ,input_types=[DoubleType(),DoubleType()]
        ,is_permanent = False
        ,name = "MARGIN_UDF" # For usage in SQL statements
        ,replace = True
    )
)
print("Non-vectorized UDF margin_udf created")

# Create a DataFrame using the non-vectorized UDF
from snowflake.snowpark.types import DecimalType
parts_margin_df_non_vectorized = (
    cost_and_price_df
        .select(
             col("*")
            ,margin_udf(col("retail_price"),col("cost")).cast(DecimalType(38,2)).as_("margin")
        )
    )
print("DataFrame parts_margin_df_non_vectorized created")

---
<a id="test_non_vectorized_udf"></a>
## 3. Test A Non-Vectorized UDF and Measure Performance

1. Bounce the warehouse to clear all warehouse data cache.
1. Note the time before our action.
1. Save our `DataFrame` named `parts_margin_df_non_vectorized` as new table (overwrite mode).
1. Note the time after our action completes.
1. Hold on to the performance info for later comparison.

In [ ]:
# Start fresh
bounce_warehouse()

# Desired table for receiving the data
non_vect_table_name = "PARTS_WITH_MARGINS_NON_VECTORIZED"

print(f"Writing table from input schema {data_schema} using a non-vectorized UDF...this might take a minute or so...",end="")

# Note the current time for performance measuring
import time
before = time.time()

# Perform the write
parts_margin_df_non_vectorized.write.mode("OVERWRITE").save_as_table(non_vect_table_name)

# Note the time after our action and calculate how long the action took to complete
duration_non_vect = time.time() - before
print("... Done!")

# Count the table rows
table_row_count = session.table(non_vect_table_name).count()

# Print the number of rows processed and the performance duration
duration_report_non_vect = f"Non-vectorized:\n\tProcessing {table_row_count:,} rows from {data_schema} took {duration_non_vect:.2f} seconds."
print(duration_report_non_vect)

# Show 5 rows of data
session.table(non_vect_table_name).show(5)

---
<a id="create_vectorized_udf"></a> 

## Create a Vectorized UDF and Use It in A Transformation

There are two ways to create and configure a Snowpark for Python Vectorized UDF.
- [Option A: Use the `vectorized` Decorator](#option_a_decorator) 
- [Option B: Use Function Attributes](#option_b_decorator)

<a id="option_a_decorator"></a>
### 4A. Option A: Use the `vectorized` Decorator 

From the Snowflake docs on [Using the `vectorized` Decorator](https://docs.snowflake.com/en/developer-guide/udf/python/udf-python-batch#using-the-vectorized-decorator):
>The `_snowflake` module is available to Python UDFs that execute within Snowflake. Import the `_snowflake` module, and use the `vectorized` decorator on your handler function. Indicate that your Python function expects to receive a `pandas.DataFrame`(s) by setting the input parameter to `pandas.DataFrame`.


```python
from _snowflake import vectorized

@vectorized(input=pandas.DataFrame)
def my_func(df):
  <function logic here>
```

However, the module `_snowflake` is not available for use *outside* of a Snowflake Virtual Warehouse. This means the `@vectorized` decorator can't be used in code developed with a notebook environment (like Jupyter), your favorite IDE (IntelliJ, VS Code, etc), or a command-line tool. We would need the handler function to be created within your Snowflake account using an in-line Python UDF as part of a `CREATE FUNCTION` statement.

From a Snowsight worksheet or other SQL script, create the UDF and decorate the handler function with `@vectorized`.

```sql
CREATE OR REPLACE FUNCTION MARGIN_VECTORIZED_UDF(DOUBLE PRICE, DOUBLE COST)
    RETURNS DOUBLE
    LANGUAGE PYTHON
    RUNTIME_VERSION = 3.8
    PACKAGES = ('pandas')
    HANDLER = 'margin_vect_func'
    AS $$
```
```python
# Imports needed    
import pandas
from _snowflake import vectorized # <-- _snowflake module available server-side

# Python function decorated with @vectorized
@vectorized(input=pandas.DataFrame)
def margin_vect_func(df:pandas.DataFrame)->pandas.DataFrame:
    return df[0]-df[1]
```
```sql
    $$;
```

The above code could be run in a worksheet or script. Or, a Snowpark developer 
could take that entire declaration above as single string and use `session.sql(...)` to create the function.

```python

create_func_string = (
"""CREATE OR REPLACE FUNCTION MARGIN_VECTORIZED_UDF(PRICE DOUBLE, COST DOUBLE)
    RETURNS DOUBLE
    LANGUAGE PYTHON
    RUNTIME_VERSION = 3.8
    PACKAGES = ('pandas')
    HANDLER = 'margin_vect_func'
    AS $$
# Imports needed    
import pandas
from _snowflake import vectorized  # <-- _snowflake module available server-side

# Python function decorated as vectorized
@vectorized(input=pandas.DataFrame)
def margin_vect_func(df:pandas.DataFrame)->pandas.DataFrame:
    return df[0]-df[1]
    
    $$;""" # end create_func_string
)

session.sql(create_func_string).show() # <-- action required
```

In [ ]:
# Create a single string for a CREATE FUNCTION statement
create_func_string = (
"""CREATE OR REPLACE FUNCTION MARGIN_VECTORIZED_UDF(PRICE DOUBLE, COST DOUBLE)
    RETURNS DOUBLE
    LANGUAGE PYTHON
    RUNTIME_VERSION = 3.8
    PACKAGES = ('pandas')
    HANDLER = 'margin_vect_func'
    AS $$
# Imports needed    
import pandas
from _snowflake import vectorized

# Python function decorated as vectorized
@vectorized(input=pandas.DataFrame)
def margin_vect_func(df:pandas.DataFrame)->pandas.DataFrame:
    return df[0]-df[1]
    
    $$;"""
)

# Execute the create function statement server-side
session.sql(create_func_string).show(1,60)

# Describe the function just created
session.sql("DESCRIBE FUNCTION MARGIN_VECTORIZED_UDF(DOUBLE,DOUBLE)").show(20, 60)

---

In order to invoke our vectorized UDF in a Snowpark `DataFrame` transformation, we will use `functions.built_in(...)`. More on this can be found in the lesson notebooks on transformations. 

<a id="steps_option_a"></a>
#### 4Aa. The Steps (Option A)

1. Use a `CREATE FUNCTION` statement to create a vectorized UDF (above).
1. Create an invocable `function` object using `functions.builtin(...)`.
1. Create a new `DataFrame` by using our vectorized UDF `function` object in a transformation. 

In [ ]:
from snowflake.snowpark.functions import builtin
from snowflake.snowpark.types import DoubleType

# Create an invokable `function` object using `functions.builtin(...)`
margin_vect_udf_func = functions.builtin("MARGIN_VECTORIZED_UDF")
print(f"The type for margin_vect_udf_func is {type(margin_vect_udf_func)}")

# Create a new `DataFrame` by using our vectorized UDF `function` object in a transformation. 
parts_margin_df_vectorized_option_a = cost_and_price_df.select(
     col("*")
    ,margin_vect_udf_func(
        col("RETAIL_PRICE") # <-- First arg to our vectorized UDF
       ,col("COST")         # <-- Second arg to our vectorized UDF
     ).cast(DecimalType(38,2)).as_("MARGIN")
)

print("\nDataFrame parts_margin_df_vectorized_option_a created with schema:")
parts_margin_df_vectorized_option_a.schema.fields   

<a id="test_option_a"></a>
#### 4Ab. Test the Vectorized UDF and Measure Performance

1. Bounce the warehouse to clear all warehouse data cache.
1. Note the time before our action;
1. Save our `DataFrame` named `parts_margin_df_vectorized` as new table (overwrite mode).
1. Note the time after our action completes.
1. Hold on to the performance info for later comparison.

In [ ]:
# Start fresh
bounce_warehouse()

# Desired table for receiving the data
vect_table_name = f"PARTS_WITH_MARGINS_VECTORIZED_OPTION_A"

print(f"Writing table {vect_table_name} \n\tfrom input schema {data_schema} using a vectorized UDF...this might take a few seconds or so...",end="")

# Note the current time for performance measuring
import time
before = time.time()

# Perform the write
parts_margin_df_vectorized_option_a.write.mode("OVERWRITE").save_as_table(vect_table_name)

# Note the time after our action and calculate how long the action took to complete
duration_vect_option_a = time.time() - before
print("... Done!\n")

# Count the table rows
table_row_count = session.table(vect_table_name).count()

# Print the number of rows processed and the performance duration
duration_report_vect_option_a = f"Vectorized UDF (option A):\n\tProcessing {table_row_count:,} rows from {data_schema} took {duration_vect_option_a:.2f} seconds."
print(duration_report_vect_option_a)

# Show 5 rows of data
session.table(vect_table_name).show(5)

---
<a id="comp_perf_option_a"></a>
#### 4Ac. Compare Vectorized Performance (Option A)

Compare the durations to see which approach gave better performance. 

In [ ]:
print(duration_report_non_vect)
print(duration_report_vect_option_a)

<a id="option_b_func_attributes"></a>
### 4B. Option B: Use Function Attributes

Because the `_snowflake` module is not available in your client development environment, a developer can set a special attribute, `_sf_vectorized_input`, on the handler function as an alternative.

```python
import pandas
def my_func(df:pandas.DataFrame) -> pandas.DataFrame:
        <use df[0], df[1], etc>
    
my_func._sf_vectorized_input = pandas.DataFrame # <-- setting a function attribute
```

**Note:** The handler function can take any number of arguments 

<a id="steps_option_b"></a>
#### 4Ba. The Steps (Option B)

1. Create a Python function that uses `pandas.DataFrame`s or `pandas.Series` objects. Like our previous function, this function will calculates the margin on an order by subtracting the item cost from the item price.
1. Add the attribute `_sf_vectorized_input= pandas.DataFrame` to the function.
1. Register it as a **vectorized** Snowflake for Python UDF using `functions.pandas_udf(...)`.
1. Create a new `DataFrame` by using our vectorized UDF in a transformation. 

In [ ]:
# Create a Python function
import pandas
def margin_vect_func(df:pandas.DataFrame): # <-- One arg to our vUDF 
        return df[0] - df[1]  # df[0] will be part price
                              # df[1] will be part cost
                              # return will be margin
                              # UDF user will send two doubles to the function

# Add the attribute _sf_vectorized_input    
margin_vect_func._sf_vectorized_input = pandas.DataFrame

print(f"Python function  margin_vect_func(DataFrame,DataFrame) created")

# Register using functions.pandas_udf(...)
from snowflake.snowpark.functions import pandas_udf
from snowflake.snowpark.types import DoubleType, PandasSeriesType, PandasDataFrameType
margin_udf = (
    pandas_udf( # <-- pandas_udf could also be used as a decorator to the margin_vect_func
         func = margin_vect_func
        ,return_type = PandasSeriesType(DoubleType())
        ,input_types = [PandasDataFrameType([DoubleType(), DoubleType()])]
        ,is_permanent = False
        ,name = "MARGIN_VECT_UDF" # For usage in SQL statements
        ,replace = True  # In case we run this cell more than once
    )
)

print(f"Vectorized UDF margin_udf create, SQL usage is {margin_udf.name}")

# Create a new DataFrame by using our vectorized UDF 
from snowflake.snowpark.types import DecimalType
parts_margin_df_vectorized_option_b = cost_and_price_df.select(
     col("*")
    ,margin_udf(
        col("RETAIL_PRICE") # <-- First arg to our vectorized UDF
       ,col("COST")         # <-- Second arg to our vectorized UDF
     ).cast(DecimalType(38,2)).as_("MARGIN")
)

print("\nDataFrame parts_margin_df_vectorized_option_b created with schema:")
parts_margin_df_vectorized_option_b.schema.fields   

---
<a id="test_option_b"></a>
#### 4Bb. Test the Vectorized UDF and Measure Performance (Option B)

1. Bounce the warehouse to clear all warehouse data cache.
1. Note the time before our action.
1. Save our `DataFrame` named `parts_margin_df_vectorized` as new table (overwrite mode).
1. Note the time after our action completes.
1. Hold on to the performance info for later comparison.

In [ ]:
# Start fresh
bounce_warehouse()

# Desired table for receiving the data
vect_table_name = f"PARTS_WITH_MARGINS_VECTORIZED_OPTION_B"

print(f"Writing table {vect_table_name} \n\tfrom input schema {data_schema} using a vectorized UDF...this might take a few seconds or so...",end="")

# Note the current time for performance measuring
import time
before = time.time()

# Perform the write
parts_margin_df_vectorized_option_b.write.mode("OVERWRITE").save_as_table(vect_table_name)

# Note the time after our action and calculate how long the action took to complete
duration_vect_option_b = time.time() - before
print("... Done!\n")

# Count the table rows
table_row_count = session.table(vect_table_name).count()

# Print the number of rows processed and the performance duration
duration_report_vect_option_b = f"Vectorized UDF (option B):\n\tProcessing {table_row_count:,} rows from {data_schema} took {duration_vect_option_b:.2f} seconds."
print(duration_report_vect_option_b)

# Show 5 rows of data
session.table(vect_table_name).show(5)

---
<a id="comp_perf_option_b"></a>
#### 4Bc. Compare Vectorized Performance (Option B)

Compare the durations to see which approach gave better performance. Several runs will of both vectorized options A and B will likely show similar performance. The takeaway here is that vectorized showed a distinct improvement over non-vectorized.  

In [ ]:
print(duration_report_non_vect)
print(duration_report_vect_option_a)
print(duration_report_vect_option_b)

---
<a id="series_instead_of_dataframe"></a>
### 4C. Using `pandas.Series` vs `pandas.DataFrame`

<a id="steps_series"></a>
#### 4Ca. The Steps (Series)

1. Create a Python function that uses pandas.Series` objects as arguments. 
1. Add the attribute `_sf_vectorized_input= pandas.Series` to the function 
1. Register it as a **vectorized** Snowflake for Python UDF using `functions.pandas_udf(...)`.
1. Create a new `DataFrame` by using our vectorized UDF in a transformation. 

In [ ]:
# Create a Python function taking argumenst of type pandas.Series
import pandas
def margin_vect_series_func(
          retail_price_series:pandas.Series # <-- First arg to our vUDF - part price
         ,cost_series        :pandas.Series # <-- Second ar to our vUDF - part cost
        ) -> pandas.Series:
        return retail_price_series - cost_series  # Return margin

# Add the attribute _sf_vectorized_input = pandas.Series
margin_vect_series_func._sf_vectorized_input = pandas.Series

print(f"Python function  margin_vect_func(Series,Series) created")

from snowflake.snowpark.functions import pandas_udf
from snowflake.snowpark.types import DoubleType
margin_series_udf = (
    pandas_udf(
         func = margin_vect_series_func
        ,return_type = DoubleType()
        ,input_types = [DoubleType(),DoubleType()]
        ,is_permanent = False
        ,name = "MARGIN_VECT_UDF" # For usage in SQL statements
        ,replace = True  # In case we run this cell more than once
    )
)
print(f"Vectorized UDF margin_udf create, SQL usage is {margin_series_udf.name}")

# Create a new DataFrame by using our vectorized UDF 
from snowflake.snowpark.types import DecimalType
parts_margin_df_vectorized_series = cost_and_price_df.select(
     col("*")
    ,margin_series_udf(
        col("RETAIL_PRICE") # <-- First arg to our vectorized UDF
       ,col("COST")         # <-- Second arg to our vectorized UDF
     ).cast(DecimalType(38,2)).as_("MARGIN")
)

print("\nDataFrame parts_margin_df_vectorized_series created with schema:")
parts_margin_df_vectorized_series.schema.fields   

---
<a id="test_series"></a>
#### 4Cb. Test the Vectorized UDF and Measure Performance (Series)

1. Bounce the warehouse to clear all warehouse data cache.
1. Note the time before our action.
1. Save our `DataFrame` named `parts_margin_df_vectorized_series` as new table (overwrite mode).
1. Note the time after our action completes.
1. Hold on to the performance info for later comparison.

In [ ]:
# Start fresh
bounce_warehouse()

# Desired table for receiving the data
vect_table_name = f"PARTS_WITH_MARGINS_VECTORIZED_SERIES"

print(f"Writing table {vect_table_name} \n\tfrom input schema {data_schema} using a vectorized UDF...this might take a few seconds or so...",end="")

# Note the current time for performance measuring
import time
before = time.time()

# Perform the write
parts_margin_df_vectorized_series.write.mode("OVERWRITE").save_as_table(vect_table_name)

# Note the time after our action and calculate how long the action took to complete
duration_vect_series = time.time() - before
print("... Done!\n")

# Count the table rows
table_row_count = session.table(vect_table_name).count()

# Print the number of rows processed and the performance duration
duration_report_vect_series = f"Vectorized UDF (Series):\n\tProcessing {table_row_count:,} rows from {data_schema} took {duration_vect_series:.2f} seconds."
print(duration_report_vect_series)

# Show 5 rows of data
session.table(vect_table_name).show(5)

---
<a id="comp_perf_series"></a>
#### 4Cc. Compare Vectorized Performance (`pandas.Series`)

Compare the durations to see which approach gave better performance. Several runs will of both vectorized options A and B as well as the series approach will likely show similar performance. Same takeaway as before: vectorized can be faster than non-vectorized. 

In [ ]:
print(duration_report_non_vect)
print(duration_report_vect_option_a)
print(duration_report_vect_option_b)
print(duration_report_vect_series)

---
<a id="AV_cleanup"></a>

## 5. Clean Up and Close Session

Best practice is to clean up demo objects, suspend our warehouse, and close the Snowpark Session object.

In [ ]:
close_session_and_clean_up(get_lesson())

### &#10071; `Shut Down Kernel`
> After completing the activities in a notebook and before moving on to the next exercise, shut down the completed notebook by right-clicking on the notebook name and selecting `Shut Down Kernel`.